In [1]:
from datasets import load_dataset
import torch
import numpy
import scipy
import diffusers
import transformers
import datasets
import huggingface_hub
import accelerate
import random
import re
from copy import deepcopy
import json

In [2]:
ds = load_dataset(
    "CSU-JPG/TextAtlas5M",
    "CleanTextSynth",
    split="train",
    streaming=True
)

small_ds = list(ds.take(10000))

Resolving data files:   0%|          | 0/67 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/67 [00:00<?, ?it/s]

In [3]:
print(small_ds[0])

{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=512x512 at 0x13CB69A10>, 'image_path': '000004f933f14f65bfcd6ee1d54d4e69.png', 'annotation': 'A 512x512 text image with the following settings: font_size 231 , font_type Winterday-jEqqO , font_color [77, 34, 70] displaying the text: the library'}


In [4]:
TEXT_MARKER = "displaying the text:"

def extract_text_from_annotation(annotation: str):
    idx = annotation.find(TEXT_MARKER)
    if idx == -1:
        return annotation, None

    prefix = annotation[: idx + len(TEXT_MARKER)]
    text = annotation[idx + len(TEXT_MARKER):].strip()
    return prefix, text

In [5]:
def substitute_chars(text):
    mapping = {
        "O": "0", "o": "0",
        "I": "1", "i": "1",
        "S": "5", "s": "5",
    }

    chars = list(text)
    idxs = [i for i, c in enumerate(chars) if c in mapping]

    if not idxs:
        return text

    i = random.choice(idxs)
    chars[i] = mapping[chars[i]]
    return "".join(chars)


def delete_chars(text):
    chars = list(text)
    idxs = [i for i, c in enumerate(chars) if not c.isspace()]

    if len(idxs) <= 1:
        return text

    i = random.choice(idxs)
    del chars[i]
    return "".join(chars)


def transpose_chars(text):
    chars = list(text)
    idxs = [i for i in range(len(chars) - 1) if not chars[i].isspace() and not chars[i + 1].isspace()]

    if not idxs:
        return text

    i = random.choice(idxs)
    chars[i], chars[i + 1] = chars[i + 1], chars[i]
    return "".join(chars)


def repeat_chars(text):
    chars = list(text)
    idxs = [i for i, c in enumerate(chars) if not c.isspace()]

    if not idxs:
        return text

    i = random.choice(idxs)
    return text[:i] + chars[i] * 2 + text[i:]


def split_or_merge_words(text):
    if " " in text and random.random() < 0.5:
        # merge
        space_positions = [i for i, c in enumerate(text) if c == " "]
        if not space_positions:
            return text
        i = random.choice(space_positions)
        return text[:i] + text[i + 1:]

    # split
    words = list(re.finditer(r"\S{2,}", text))
    if not words:
        return text

    w = random.choice(words)
    pos = random.randint(w.start() + 1, w.end() - 1)
    return text[:pos] + " " + text[pos:]

In [6]:
CORRUPTION_FUNCS = [
    delete_chars,
    substitute_chars,
    transpose_chars,
    repeat_chars,
    split_or_merge_words,
]

def corrupt_text(text, apply_prob=0.4, max_ops=2):
    # ~40% семплів будуть псовані
    if random.random() > apply_prob:
        return text, []

    corrupted = text
    applied = []

    n_ops = random.randint(1, max_ops)

    for _ in range(n_ops):
        func = random.choice(CORRUPTION_FUNCS)
        new_text = func(corrupted)

        if new_text != corrupted:
            corrupted = new_text
            applied.append(func.__name__)

    return corrupted, applied

In [7]:
def corrupt_annotation(annotation, apply_prob=0.4, max_ops=2):
    prefix, text = extract_text_from_annotation(annotation)

    if text is None:
        return annotation, None, None, []

    corrupted_text, ops = corrupt_text(
        text=text,
        apply_prob=apply_prob,
        max_ops=max_ops
    )

    new_annotation = f"{prefix} {corrupted_text}"
    return new_annotation, text, corrupted_text, ops

In [8]:
corrupted_ds = []

for idx, sample in enumerate(small_ds):
    new_ann, original_text, corrupted_text, ops = corrupt_annotation(
        sample["annotation"],
        apply_prob=0.4,
        max_ops=2
    )

    row = {
        "sample_id": idx,
        "image_path": sample["image_path"],   # ID / шлях картинки
        "image": sample["image"],             # сам PIL image, якщо хочеш тримати в RAM
        "annotation": sample["annotation"],
        "corrupted_annotation": new_ann,
        "original_text": original_text,
        "corrupted_text": corrupted_text,
        "ops": ops,
        "is_corrupted": len(ops) > 0,
    }

    corrupted_ds.append(row)

print("Done:", len(corrupted_ds))
print(corrupted_ds[0].keys())

Done: 10000
dict_keys(['sample_id', 'image_path', 'image', 'annotation', 'corrupted_annotation', 'original_text', 'corrupted_text', 'ops', 'is_corrupted'])


In [9]:
num_corrupted = sum(x["is_corrupted"] for x in corrupted_ds)
ratio = num_corrupted / len(corrupted_ds)

print("Corrupted samples:", num_corrupted)
print("Corrupted ratio:", round(ratio, 4))

Corrupted samples: 3994
Corrupted ratio: 0.3994


In [10]:
shown = 0
for row in corrupted_ds:
    if row["is_corrupted"]:
        print("sample_id:", row["sample_id"])
        print("image_path:", row["image_path"])
        print("ORIGINAL TEXT:", row["original_text"])
        print("CORRUPTED TEXT:", row["corrupted_text"])
        print("OPS:", row["ops"])
        print("-" * 70)

        shown += 1
        if shown == 10:
            break

sample_id: 0
image_path: 000004f933f14f65bfcd6ee1d54d4e69.png
ORIGINAL TEXT: the library
CORRUPTED TEXT: t he library
OPS: ['split_or_merge_words']
----------------------------------------------------------------------
sample_id: 2
image_path: 000008301fde4e0bb8a20410b9872af4.png
ORIGINAL TEXT: which charted at No.
CORRUPTED TEXT: which chared at No.
OPS: ['delete_chars']
----------------------------------------------------------------------
sample_id: 5
image_path: 000010d62a894119aec4b21a34915427.png
ORIGINAL TEXT: Many of her pictures gained honors as well as prizes. In 1909 she was awarded the first prize at the woman's exhibit of the Artists' Guild. This was "An Autumn Landscape".
CORRUPTED TEXT: Many of her pictures gained honors as well as pr izes. In 1909 she was awarded the first prize at the woman's exhibit of the Artists' Guild. This was "An Autumn Landscape".
OPS: ['split_or_merge_words']
----------------------------------------------------------------------
sample_id: 6
im

In [11]:
export_rows = []

for row in corrupted_ds:
    export_rows.append({
        "sample_id": row["sample_id"],
        "image_path": row["image_path"],
        "annotation": row["annotation"],
        "corrupted_annotation": row["corrupted_annotation"],
        "original_text": row["original_text"],
        "corrupted_text": row["corrupted_text"],
        "ops": row["ops"],
        "is_corrupted": row["is_corrupted"],
    })

with open("corrupted_textatlas_10k.jsonl", "w", encoding="utf-8") as f:
    for row in export_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved to corrupted_textatlas_10k.jsonl")

Saved to corrupted_textatlas_10k.jsonl
